# 🚀 01: EvictingCache Core, Compaction, and Streaming Decode

*Part of the KV-Cache Eviction Capstone Series*
*Estimated time: 10–15 minutes*

---

Welcome to the engine room. In this notebook, we validate the custom `EvictingCache` implementation before we trust it with benchmark sweeps. 

We will test logical-to-physical position mappings, memory compaction, policy budget enforcement, full-cache mathematical equivalence, and streaming decode after a compaction boundary.

### Experiment Roadmap
```text
[Prompt] → [EvictingCache] → [Policy] → [Compaction] → [Streaming decode]
```


## Step 1: Why Does This Matter?

If we evict tokens but corrupt the position IDs, the model's attention mechanism will hallucinate. If we exceed the memory budget, the system will crash in production. 

We must prove the mechanism works. Our **required assertions** are:
- Full-cache execution matches an ordinary dynamic-cache reference exactly.
- Every compressed policy keeps the physical cache size strictly at or below its budget.
- Logical positions remain strictly monotonic after physical compaction.
- A streaming decode step successfully continues generation even after an eviction boundary has shifted the physical memory.


In [ ]:
NOTEBOOK_ID = '01_evicting_cache_core_tests'
REQUESTED_PROFILE = 't4'

# Colab bootstrap: install pinned dependencies and unpack the shared core.
# Upload kvcore_bundle.zip supplied with this notebook suite if kvcore is not present.
from pathlib import Path
import sys, subprocess, zipfile

PINNED = [
    'transformers==4.56.2', 'accelerate==1.10.1', 'datasets==4.0.0',
    'huggingface_hub==0.34.4', 'bitsandbytes==0.47.0', 'safetensors==0.6.2',
    'sentencepiece==0.2.1', 'scipy==1.16.1', 'matplotlib==3.10.6',
    'seaborn==0.13.2', 'pandas==2.3.2',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PINNED])

if not Path('kvcore').exists():
    try:
        from google.colab import files
        print('Upload kvcore_bundle.zip from the delivered suite.')
        uploaded = files.upload()
        archive = next((Path(name) for name in uploaded if name.endswith('.zip')), None)
        if archive is None:
            raise FileNotFoundError('Please upload kvcore_bundle.zip.')
        with zipfile.ZipFile(archive) as zf:
            zf.extractall('.')
    except ImportError as error:
        raise RuntimeError('Run in Google Colab or place the kvcore directory beside this notebook.') from error

sys.path.insert(0, str(Path('.').resolve()))
from kvcore import *
from kvcore.config import BENCHMARKS, MODELS, POLICY_DEFAULTS, PROFILES, SUITE_VERSION
print({'suite_version': SUITE_VERSION, 'ruler_revision': BENCHMARKS['ruler']['revision'], 'longbench_revision': BENCHMARKS['longbench']['revision']})


In [ ]:
import json, os, platform, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

REQUESTED_PROFILE = REQUESTED_PROFILE  # defined by the notebook title cell
NOTEBOOK_ID = globals().get('NOTEBOOK_ID', 'runtime')
set_all_seeds(590)
profile, runtime_status = select_profile(REQUESTED_PROFILE)
run_root = ensure_run_root(f'kv_eviction_{NOTEBOOK_ID}')
manifest = run_manifest(
    run_root,
    notebook=NOTEBOOK_ID,
    requested_profile=REQUESTED_PROFILE,
    active_profile=profile.name,
    model=model_spec(profile.model_tier),
    runtime_status=runtime_status,
)
print(json.dumps({'notebook': NOTEBOOK_ID, 'requested_profile': REQUESTED_PROFILE, 'active_profile': profile.name, 'runtime_status': runtime_status, 'run_root': str(run_root)}, indent=2))
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU runtime is required for model execution. In Colab: Runtime > Change runtime type > GPU.')
print('GPU:', torch.cuda.get_device_name(0), 'VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))


In [ ]:
# Model and tokenizer are always loaded at immutable Hub revisions from kvcore.config.
# If the runtime is smaller than the requested tier, runtime_status records the fallback.
model, tokenizer = load_model_and_tokenizer(model_spec(profile.model_tier), load_mode=profile.load_mode, attn_implementation='sdpa')
print({'model': model.config._name_or_path, 'requested_profile': REQUESTED_PROFILE, 'active_profile': profile.name, 'load_mode': profile.load_mode})


In [ ]:
# All policies use the same per-example cache budget. Full cache is the uncompressed reference.
POLICY_SPECS = [
    {'type': 'full'}, {'type': 'fifo', 'min_recent': 32},
    {'type': 'random', 'seed': 590, 'min_recent': 32}, {'type': 'uniform', 'min_recent': 32},
    {'type': 'sink_recent', 'sink_tokens': 4, 'min_recent': 32},
    {'type': 'attention_topk', 'sink_tokens': 4, 'min_recent': 32},
    {'type': 'h2o', 'recent_fraction': 0.50, 'sink_tokens': 4, 'min_recent': 32},
]


In [ ]:
from kvcore.cache import prefill, greedy_decode_step
from kvcore.policies import CacheState, policy_from_spec

prompt_ids = tokenizer('A cache should retain important facts while preserving the newest context. ' * 60, return_tensors='pt').input_ids.to(next(model.parameters()).device)
cache, out = prefill(model, prompt_ids, chunk_size=min(256, prompt_ids.shape[-1]), output_attentions=False)
state = CacheState(positions=cache.physical_positions, step=1)
print({'logical_length': cache.logical_length, 'physical_length': cache.physical_length, 'positions_tail': cache.physical_positions[-8:].detach().cpu().tolist()})


## Step 2: Cache Compaction and Policy-Budget Tests

Let us apply every configured eviction policy to a test prompt and verify that the resulting physical cache length respects the requested budget, while keeping logical positions intact.


In [ ]:
checks = []
for spec in POLICY_SPECS:
    policy = policy_from_spec(spec)
    test_cache, _ = prefill(model, prompt_ids, chunk_size=min(256, prompt_ids.shape[-1]), output_attentions=False)
    local_state = CacheState(positions=test_cache.physical_positions, step=1)
    budget = min(128, max(64, test_cache.physical_length // 2))
    info = test_cache.apply_policy(policy, local_state, budget=budget, reason='notebook_01_unit_test')
    assert_policy_respects_budget(test_cache, budget, policy.name)
    positions = test_cache.physical_positions.detach().cpu().tolist()
    assert positions == sorted(positions), f'nonmonotonic logical positions for {policy.name}'
    checks.append({'policy': policy.name, 'budget': budget, 'physical_length': test_cache.physical_length, **info})
checks_df = pd.DataFrame(checks); checks_df.to_csv(run_root / 'results' / 'policy_budget_checks.csv', index=False); display(checks_df)


## Step 3: Full-Cache Equivalence and Post-Compaction Decode

Finally, we must prove that our custom cache does not alter the model's output when no tokens are evicted, and that it can successfully stream new tokens *after* an eviction event.

### What to Look For
This test uses a deterministic greedy next-token comparison. If it passes without an assertion error, our cache is mathematically sound and ready for the T4 benchmark sweeps.


In [ ]:
# Reference and full policy must expose the same greedy next token before any compaction.
reference_cache, reference_out = prefill(model, prompt_ids[:, :min(256, prompt_ids.shape[-1])], chunk_size=128, output_attentions=False)
full_cache, full_out = prefill(model, prompt_ids[:, :min(256, prompt_ids.shape[-1])], chunk_size=128, output_attentions=False)
assert_full_cache_equivalence(reference_out.logits[:, -1, :], full_out.logits[:, -1, :])

# Streaming decode after a forced compaction boundary.
compact_cache, compact_out = prefill(model, prompt_ids[:, :min(384, prompt_ids.shape[-1])], chunk_size=128, output_attentions=False)
compact_policy = policy_from_spec({'type': 'sink_recent', 'sink_tokens': 4, 'min_recent': 32})
compact_cache.apply_policy(compact_policy, CacheState(positions=compact_cache.physical_positions, step=1), budget=min(128, compact_cache.physical_length), reason='pre_stream')
token = compact_out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
trace = []
for step in range(3):
    compact_cache, token, logits = greedy_decode_step(model, compact_cache, token, logical_position=compact_cache.logical_length, output_attentions=False)
    trace.append({'step': step, 'logical_length': compact_cache.logical_length, 'physical_length': compact_cache.physical_length, 'token_id': int(token.item())})
assert len(trace) == 3
pd.DataFrame(trace).to_csv(run_root / 'results' / 'streaming_decode_after_compaction.csv', index=False)
display(pd.DataFrame(trace))


In [ ]:
write_json(run_root / 'checks' / 'cache_core_complete.json', {'status': 'passed', 'checks': checks})
print('Cache core checks passed:', run_root)
